Tip: Welcome to the Investigate a Dataset project! You will find tips in quoted sections like this to help organize your approach to your investigation. Once you complete this project, remove these Tip sections from your report before submission. First things first, you might want to double-click this Markdown cell and change the title so that it reflects your dataset and investigation.

Project: Investigate a Dataset - [TMDb Movies]
Table of Contents
Introduction
Data Wrangling
Exploratory Data Analysis
Conclusions

Introduction
Dataset Description
El conjunto de datos seleccionado para este proyecto contiene información sobre aproximadamente 10.000 películas recopiladas de The Movie Database (TMDb). El dataset incluye variables relacionadas con los ingresos, el presupuesto, la popularidad, la duración y las valoraciones de los usuarios, entre otros aspectos.

Cada fila del conjunto de datos representa una película individual. Algunas columnas, como cast y genres, contienen múltiples valores separados por el carácter |. Aunque estas columnas contienen ciertos caracteres inconsistentes, no se realizará una limpieza exhaustiva de ellas, ya que no son el foco principal de este análisis.

El dataset también incluye columnas que terminan en _adj, las cuales representan el presupuesto y los ingresos ajustados por inflación a dólares del año 2010. Estas variables permiten realizar comparaciones más consistentes entre películas estrenadas en distintos años.

Entre las principales columnas del conjunto de datos se encuentran:

id: Identificador único de la película
original_title: Título original de la película
budget: Presupuesto de producción
budget_adj: Presupuesto ajustado por inflación
revenue: Ingresos obtenidos
revenue_adj: Ingresos ajustados por inflación
popularity: Medida de popularidad según TMDb
runtime: Duración de la película en minutos
vote_average: Valoración promedio de los usuarios
vote_count: Número total de votos
release_date: Fecha de estreno
genres: Géneros asociados a la película
Question(s) for Analysis
El objetivo principal de este análisis es identificar qué características están asociadas con el éxito económico de una película, medido a través de los ingresos ajustados por inflación (revenue_adj).

Las preguntas que se abordarán en este informe son las siguientes:

¿Existe una relación entre el presupuesto ajustado de una película (budget_adj) y sus ingresos ajustados (revenue_adj)?
¿Las películas con mayor popularidad tienden a generar mayores ingresos?
¿La valoración promedio de los usuarios (vote_average) se asocia con mayores ingresos?
¿Qué géneros cinematográficos han sido más populares a lo largo de los años?
En este análisis, la variable dependiente principal es revenue_adj, mientras que budget_adj, popularity, vote_average y genres actúan como variables independientes.

In [ ]:
# Import required libraries for data analysis
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Display plots inline in the notebook
%matplotlib inline

Data Wrangling
Tip: In this section of the report, you will load in the data, check for cleanliness, and then trim and clean your dataset for analysis. Make sure that you document your data cleaning steps in mark-down cells precisely and justify your cleaning decisions.

General Properties
Tip: You should not perform too many operations in each cell. Create cells freely to explore your data. One option that you can take with this project is to do a lot of explorations initially. This does not have to be organized, but make sure you use enough comments to understand the purpose of each code cell. Then, after you're done with your analysis, trim the excess and organize your steps so that you have a flowing, cohesive report.

Data Wrangling
En esta sección se carga el conjunto de datos, se exploran sus propiedades generales y se realizan los pasos necesarios para limpiarlo y prepararlo para el análisis exploratorio.
Se documentan y justifican todas las decisiones de limpieza realizadas. ``

In [ ]:
# Load the TMDb movies dataset
df = pd.read_csv('tmdb-movies.csv')

# Display the first few rows
df.head()
# Check the dimensions of the dataset
df.shape

In [ ]:
# El dataset contiene aproximadamente 10.000 filas y 21 columnas.
# Display concise summary of the dataframe
df.info()



In [ ]:
# Summary statistics for numerical columns
df.describe()

In [ ]:
# Count missing values in each column
df.isnull().sum()

Initial Observations
A partir de la exploración inicial del conjunto de datos se observan los siguientes puntos importantes:

Varias columnas contienen valores faltantes, especialmente homepage, tagline y keywords.
Las columnas financieras budget, revenue, budget_adj y revenue_adj contienen muchos valores iguales a cero, lo cual indica datos no reportados en lugar de valores reales.
Las columnas genres, cast y production_companies contienen múltiples valores separados por el carácter |.
La variable release_date está en formato texto y no se usará directamente en el análisis, ya que se dispone de la columna release_year.
Estas observaciones guiarán las decisiones de limpieza realizadas a continuación.

In [ ]:
# Select relevant columns for analysis
columns_of_interest = [
    'original_title',
    'release_year',
    'popularity',
    'vote_average',
    'vote_count',
    'budget_adj',
    'revenue_adj',
    'genres'
]

df_clean = df[columns_of_interest].copy()

df_clean.head()

In [ ]:
# Remove rows with zero or missing adjusted budget or revenue
df_clean = df_clean[
    (df_clean['budget_adj'] > 0) &
    (df_clean['revenue_adj'] > 0)
]

df_clean.shape

In [ ]:
# Check missing values after cleaning
df_clean.isnull().sum()

Data Cleaning
Tip: Make sure that you keep your reader informed on the steps that you are taking in your investigation. Follow every code cell, or every set of related code cells, with a markdown cell to describe to the reader what was found in the preceding cell(s). Try to make it so that the reader can then understand what they will be seeing in the following cell(s).

Data Cleaning Summary
Se realizaron los siguientes pasos de limpieza y preparación de los datos:

Se seleccionaron únicamente las columnas relevantes para el análisis.
Se eliminaron las películas con valores faltantes o iguales a cero en budget_adj y revenue_adj, ya que no proporcionan información financiera útil.
No se realizó una limpieza profunda de las columnas categóricas multivaluadas (genres), ya que se utilizarán únicamente para análisis descriptivos.
El conjunto de datos resultante está listo para el análisis exploratorio.

Data Cleaning
Después de explorar la estructura general del conjunto de datos y detectar posibles problemas, en esta sección se llevan a cabo los pasos necesarios para limpiar el dataset.
El objetivo es obtener un conjunto de datos consistente y adecuado para el análisis exploratorio, manteniendo únicamente la información relevante para responder las preguntas planteadas.

In [ ]:
def clean_movie_data(df, columns):
    """
    Cleans the movie dataset by:
    - Selecting relevant columns
    - Removing rows with zero or missing financial data
    """
    # Select relevant columns
    df_cleaned = df[columns].copy()
    
    # Remove invalid financial data
    df_cleaned = df_cleaned[
        (df_cleaned['budget_adj'] > 0) &
        (df_cleaned['revenue_adj'] > 0)
    ]
    
    return df_cleaned

Data Issues Identified
A partir de la exploración inicial, se identificaron los siguientes problemas principales:

Valores faltantes representados como 0 en las columnas financieras (budget_adj y revenue_adj).
Columnas con un alto número de valores nulos que no son relevantes para el análisis (por ejemplo, homepage o tagline).
Variables categóricas con múltiples valores en una sola celda, separadas por el carácter |.
En los siguientes pasos se aplicarán las decisiones de limpieza necesarias para abordar estos problemas.

In [ ]:
columns_of_interest = [
    'original_title',
    'release_year',
    'popularity',
    'vote_average',
    'vote_count',
    'budget_adj',
    'revenue_adj',
    'genres'
]

df_clean = clean_movie_data(df, columns_of_interest)
df_clean.head()

In [ ]:
# Reusing the cleaning function on a subset of the data
df_recent = df[df['release_year'] >= 2010]
df_recent_clean = clean_movie_data(df_recent, columns_of_interest)

df_recent_clean.shape

In [ ]:
# Check missing values after cleaning
df_clean.isnull().sum()

Data Cleaning Summary
Se realizaron los siguientes pasos de limpieza y preparación de los datos:

Se seleccionaron únicamente las columnas relevantes para el análisis.
Se eliminaron las películas con valores faltantes o iguales a cero en budget_adj y revenue_adj, ya que no proporcionan información financiera útil.
No se realizó una limpieza profunda de las columnas categóricas multivaluadas (genres), ya que se utilizarán únicamente para análisis descriptivos.
El conjunto de datos resultante está listo para el análisis exploratorio.


Exploratory Data Analysis
Tip: Now that you've trimmed and cleaned your data, you're ready to move on to exploration. Compute statistics and create visualizations with the goal of addressing the research questions that you posed in the Introduction section. You should compute the relevant statistics throughout the analysis when an inference is made about the data. Note that at least two or more kinds of plots should be created as part of the exploration, and you must compare and show trends in the varied visualizations. Remember to utilize the visualizations that the pandas library already has available.

Tip: Investigate the stated question(s) from multiple angles. It is recommended that you be systematic with your approach. Look at one variable at a time, and then follow it up by looking at relationships between variables. You should explore at least three variables in relation to the primary question. This can be an exploratory relationship between three variables of interest, or looking at how two independent variables relate to a single dependent variable of interest. Lastly, you should perform both single-variable (1d) and multiple-variable (2d) explorations.

Research Question 1 (Replace this header name!)
Exploratory Data Analysis
En esta sección se realiza un análisis exploratorio del conjunto de datos ya limpio, con el objetivo de responder a las preguntas de investigación planteadas.
Se utilizarán estadísticas descriptivas y visualizaciones para identificar patrones, tendencias y relaciones entre las variables, evitando conclusiones causales.

In [ ]:
### Research Question 1: What factors are associated with higher adjusted movie revenue?
# Distribution of adjusted revenue
df_clean['revenue_adj'].describe()

In [ ]:
# Histogram of adjusted revenue
df_clean['revenue_adj'].plot(
    kind='hist',
    bins=30,
    figsize=(8,6),
    title='Distribution of Adjusted Revenue'
)

La distribución de los ingresos ajustados muestra una fuerte asimetría positiva. La mayoría de las películas obtienen ingresos relativamente bajos, mientras que unas pocas generan ingresos excepcionalmente altos.

In [ ]:
# Scatter plot: budget vs revenue
df_clean.plot(
    x='budget_adj',
    y='revenue_adj',
    kind='scatter',
    figsize=(8,6),
    title='Adjusted Budget vs Adjusted Revenue'
)

Existe una relación positiva entre el presupuesto ajustado y los ingresos ajustados. Las películas con mayor presupuesto tienden a generar mayores ingresos, aunque se observa una variabilidad considerable.

In [ ]:
# Scatter plot: popularity vs revenue
df_clean.plot(
    x='popularity',
    y='revenue_adj',
    kind='scatter',
    figsize=(8,6),
    title='Popularity vs Adjusted Revenue'

Se observa que niveles más altos de popularidad suelen asociarse con mayores ingresos ajustados.
Sin embargo, la relación no es estrictamente lineal y existen películas populares con ingresos moderados.

### Research Question 2  (Replace this header name!)

In [ ]:
### Research Question 2: How do user ratings relate to movie financial performance?
# Distribution of vote averages
df_clean['vote_average'].describe()


In [ ]:
# Histogram of vote average
df_clean['vote_average'].plot(
    kind='hist',
    bins=20,
    figsize=(8,6),
    title='Distribution of Average Ratings'
)

Las calificaciones promedio se concentran principalmente entre valores de 6 y 7. Esto indica que la mayoría de las películas reciben evaluaciones moderadamente positivas.

In [ ]:
# Scatter plot: vote average vs revenue
df_clean.plot(
    x='vote_average',
    y='revenue_adj',
    kind='scatter',
    figsize=(8,6),
    title='Average Rating vs Adjusted Revenue'
)

La relación entre la calificación promedio y los ingresos ajustados es débil. Películas con calificaciones altas no necesariamente generan mayores ingresos, lo que sugiere que otros factores juegan un papel más relevante en el éxito financiero.

In [ ]:
# Scatter plot: vote count vs revenue
df_clean.plot(
    x='vote_count',
    y='revenue_adj',
    kind='scatter',
    figsize=(8,6),
    title='Vote Count vs Adjusted Revenue'
)

Se observa que las películas con mayor número de votos tienden a generar mayores ingresos ajustados. Esto puede reflejar una mayor exposición o alcance del público.

NumPy Array Manipulation and Vectorized Operations
In this section, NumPy arrays are explicitly used to manipulate and analyze movie revenue data. Vectorized NumPy operations are applied instead of Python loops to improve computational efficiency and to satisfy the project requirement of using NumPy arrays alongside pandas Series and DataFrames.

In [ ]:
# Convert adjusted revenue column to a NumPy array
revenue_np = df_clean['revenue_adj'].to_numpy()

# Calculate revenue threshold using NumPy
revenue_mean = np.mean(revenue_np)

# Create a NumPy boolean mask using vectorized operations
high_revenue_mask = revenue_np > revenue_mean

# Count movies above and below the mean using NumPy
movies_above_mean = np.sum(high_revenue_mask)
movies_below_mean = np.sum(~high_revenue_mask)

movies_above_mean, movies_below_mean

In [ ]:
# Categorize movies using NumPy vectorization
revenue_category = np.where(
    revenue_np >= revenue_mean,
    'High Revenue',
    'Low Revenue'
)

# Add NumPy result back to the DataFrame
df_clean['revenue_category'] = revenue_category

df_clean[['revenue_adj', 'revenue_category']].head()

Using NumPy vectorized operations, movies were categorized based on whether their adjusted revenue is above or below the mean. This NumPy-generated categorization was then integrated back into the pandas DataFrame to continue the analysis.

Conclusions
A partir del análisis exploratorio se obtuvieron los siguientes hallazgos principales:

Las películas con mayor presupuesto ajustado tienden a generar mayores ingresos, aunque la relación presenta variabilidad.
La popularidad muestra una asociación positiva con los ingresos ajustados, pero no garantiza el éxito financiero.
Las calificaciones promedio de los usuarios no presentan una relación fuerte con los ingresos, indicando que una buena valoración no implica necesariamente mayor rentabilidad.
El número de votos parece estar más relacionado con el éxito financiero que la calificación promedio.
Limitations
Este análisis presenta varias limitaciones:

No se realizaron pruebas estadísticas formales, por lo que no se pueden afirmar relaciones causales.
El conjunto de datos contiene valores financieros faltantes que fueron eliminados, lo cual podría sesgar los resultados.
No se consideraron factores externos como marketing, fecha de estreno, competencia o contexto cultural.
Análisis adicionales que integren más variables o métodos estadísticos podrían ofrecer una comprensión más profunda del éxito cinematográfico. ``


Conclusions
Tip: Finally, summarize your findings and the results that have been performed in relation to the question(s) provided at the beginning of the analysis. Summarize the results accurately, and point out where additional research can be done or where additional information could be useful.

Tip: Make sure that you are clear with regards to the limitations of your exploration. You should have at least 1 limitation explained clearly.

Tip: If you haven't done any statistical tests, do not imply any statistical conclusions. And make sure you avoid implying causation from correlation!

Tip: Once you are satisfied with your work here, check over your report to make sure that it is satisfies all the areas of the rubric (found on the project submission page at the end of the lesson). You should also probably remove all of the "Tips" like this one so that the presentation is as polished as possible.

Submitting your Project
Tip: Before you submit your project, you need to create a .html or .pdf version of this notebook in the workspace here. To do that, run the code cell below. If it worked correctly, you should see output that starts with NbConvertApp] Converting notebook, and you should see the generated .html file in the workspace directory (click on the orange Jupyter icon in the upper left).

Tip: Alternatively, you can download this report as .html via the File > Download as submenu, and then manually upload it into the workspace directory by clicking on the orange Jupyter icon in the upper left, then using the Upload button.

Tip: Once you've done this, you can submit your project by clicking on the "Submit Project" button in the lower right here. This will create and submit a zip file with this .ipynb doc and the .html or .pdf version you created. Congratulations!

In [ ]:
# Running this cell will execute a bash command to convert this notebook to an .html file
!python -m nbconvert --to html Investigate_a_Dataset.ipynb